In [2]:
%%writefile preprocesamiento.py
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path

# --- Rutas ---
BASE_DIR = Path(__file__).parent / "data" if '__file__' in globals() else Path("data")
BASE_DIR.mkdir(exist_ok=True)

RUTA_MAMO_XLSX = BASE_DIR / "Mamografos_RENIPRESS_Merge_FIX.xlsx"
RUTA_POB_XLSX = BASE_DIR / "afiliadas_40_69_por_domicilio.xlsx"
URL_GEOJSON_DISTRITAL = "https://github.com/juaneladio/peru-geojson/raw/master/peru_distrital_simple.geojson"

RUTA_SALIDA_MAMO = BASE_DIR / "mamografos_procesado.geojson"
RUTA_SALIDA_POB = BASE_DIR / "afiliadas_40_69_procesado.parquet"
RUTA_SALIDA_DISTRITAL = BASE_DIR / "peru_distrital_simple.geojson"

def procesar_mamografos() -> gpd.GeoDataFrame:
    print("Procesando base de mamógrafos...")
    df_mamo = pd.read_excel(RUTA_MAMO_XLSX)
    df_mamo["UBIGEO_STR"] = df_mamo["RENIPRESS_UBIGEO"].astype(str).str.zfill(6)
    df_mamo["RENIPRESS_NORTE"] = pd.to_numeric(df_mamo["RENIPRESS_NORTE"], errors="coerce")
    df_mamo["RENIPRESS_ESTE"] = pd.to_numeric(df_mamo["RENIPRESS_ESTE"], errors="coerce")
    df_mamo = df_mamo.dropna(subset=["RENIPRESS_NORTE", "RENIPRESS_ESTE"])

    geom = [Point(xy) for xy in zip(df_mamo["RENIPRESS_ESTE"], df_mamo["RENIPRESS_NORTE"])]
    gdf_mamo = gpd.GeoDataFrame(df_mamo, geometry=geom, crs="EPSG:4326")

    columnas_utiles = ["UBIGEO_STR", "RENIPRESS_NOMBRE", "CATEGORIA", "N_MAMOGRAFOS", "geometry"]
    columnas_existentes = [c for c in columnas_utiles if c in gdf_mamo.columns]
    gdf_mamo = gdf_mamo[columnas_existentes]

    print(f"  -> {len(gdf_mamo)} centros con mamógrafo procesados.")
    return gdf_mamo

def procesar_poblacion() -> pd.DataFrame:
    print("Procesando base de demanda poblacional...")
    df_pob = pd.read_excel(RUTA_POB_XLSX)
    df_pob["UBIGEO_STR"] = df_pob["UBIGEO"].astype(str).str.zfill(6)

    columnas_utiles = ["DISTRITO", "UBIGEO_STR", "TOTAL_AFILIADOS"]
    columnas_existentes = [c for c in columnas_utiles if c in df_pob.columns]
    df_pob = df_pob[columnas_existentes]

    print(f"  -> {len(df_pob)} registros de demanda poblacional procesados.")
    return df_pob

def descargar_capa_distrital() -> gpd.GeoDataFrame:
    print("Descargando capa distrital de Perú (solo esta vez)...")
    peru_distritos = gpd.read_file(URL_GEOJSON_DISTRITAL)
    print(f"  -> {len(peru_distritos)} distritos descargados.")
    return peru_distritos

def main():
    gdf_mamo = procesar_mamografos()
    gdf_mamo.to_file(RUTA_SALIDA_MAMO, driver="GeoJSON")
    print(f"Guardado: {RUTA_SALIDA_MAMO}")

    df_pob = procesar_poblacion()
    df_pob.to_parquet(RUTA_SALIDA_POB, index=False)
    print(f"Guardado: {RUTA_SALIDA_POB}")

    peru_distritos = descargar_capa_distrital()
    peru_distritos.to_file(RUTA_SALIDA_DISTRITAL, driver="GeoJSON")
    print(f"Guardado: {RUTA_SALIDA_DISTRITAL}")

    print("\n✅ Preprocesamiento completo.")

if __name__ == "__main__":
    main()

Writing preprocesamiento.py


In [3]:
%run preprocesamiento.py

Procesando base de mamógrafos...
  -> 62 centros con mamógrafo procesados.
Guardado: c:\Users\arian\Downloads\geo-agent-mamografos\geo-agent-mamografos\data\mamografos_procesado.geojson
Procesando base de demanda poblacional...
  -> 1891 registros de demanda poblacional procesados.
Guardado: c:\Users\arian\Downloads\geo-agent-mamografos\geo-agent-mamografos\data\afiliadas_40_69_procesado.parquet
Descargando capa distrital de Perú (solo esta vez)...
  -> 1834 distritos descargados.
Guardado: c:\Users\arian\Downloads\geo-agent-mamografos\geo-agent-mamografos\data\peru_distrital_simple.geojson

✅ Preprocesamiento completo.
